# Fine-tuning For Item Development

## What is fine-tuning
Fine-tuning is a process that takes a model that has already been trained on a large general corpus and continues its training on a small set of your own examples, so that its weights shift toward producing what those examples look like. 

## When to fine-tune

In assessment, fine-tuning tends to be treated as the serious option, probably the thing you do once prompting is beneath you. That leads programs to spend a lot of time and energy on a fine-tune that a schema and fewshot samples would have handled perfectly well.

**What fine-tuning buys you**

- House style: Stem phrasing conventions, option parallelism, rationale voice.
- Format compliance: Output that is in your format nearly every time, without spending hundreds to thousands of tokens on every call.
- Cheaper inference at volume: If you are generating tens of thousands of drafts, a tuned 0.6B model that produces your format is cheaper than a frontier model plus a long prompt.
- Deployability: A small open model you can run on your own hardware, or inside a cloud tenant you govern.

**What fine-tuning does not buy you**

- Knowledge: Training on 200 items does not teach a model your curriculum. It teaches it what your items look like.
- Factual correction: You cannot patch a wrong fact by fine-tuning around it.
- Judgment: A tuned model is not a reviewer. It has no idea whether an item is defensible.

## Ways to fine-tune

| Approach | What is trained | Cost | When it fits |
|---|---|---|---|
| Full fine-tuning | Every weight in the model | Several times the model's size in GPU memory; a new copy of the whole model; on small data, loss of prior knowledge | Thousands of examples and a GPU budget |
| PEFT: adapters | Small layers inserted into the model | Low memory; small file; slight inference slowdown | Historical; superseded by LoRA |
| PEFT: prompt tuning | A few learned tokens prepended to the input | Very few parameters | Tasks where the prompt, not the output style, is what needs to change |
| PEFT: LoRA | A low-rank correction beside selected weight matrices | Low memory; a file of tens of MB; no inference cost | Format and style on 100 to 500 examples. |
| PEFT: QLoRA | LoRA with the base model stored in 4-bit | A quarter of the base memory | Fitting a 7B or 8B model on a free GPU |
| Hosted | Whatever the provider (OpenAI, Google, or a cloud platform such as Azure or Bedrock) runs on its own model | No infrastructure; per-token fees; no weights | When the provider's model and data terms are acceptable |

## 1. LoRA

| Hyperparameter | What it is | Guidance |
|---|---|---|
| `r` (rank) | Width of the inserted matrices, i.e. the adapter's capacity | 8 to 16 for style and format. Go higher only if the task is more complex; higher `r` overfits small datasets faster. |
| `lora_alpha` | Scaling applied to the adapter's contribution | Common practice is `alpha = 2 * r`. Raising it makes the adapter's influence stronger. |
| `lora_dropout` | Dropout inside the adapter during training | 0.05 to 0.1 on small datasets. This is your main regularizer when you only have about 120 examples. |
| `target_modules` | Which weight matrices get an adapter | Attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`) are the usual minimum. Adding the MLP projections (`gate_proj`, `up_proj`, `down_proj`) gives more capacity for stylistic change. |

In [ ]:
# STEP 1 of 9: install the libraries. 
%pip uninstall -q -y torchao
%pip install -q -U trl peft transformers datasets accelerate openpyxl
print("Install finished.")

In [ ]:
# STEP 2 of 9: confirm there is a GPU. Training on CPU takes hours, not minutes.
import torch

if not torch.cuda.is_available():
    raise SystemExit("No GPU found. Runtime > Change runtime type > T4 GPU > Save, then rerun from STEP 1.")

USE_BF16 = torch.cuda.get_device_capability(0)[0] >= 8    # bf16 needs a newer GPU; a T4 uses fp16
print("GPU:", torch.cuda.get_device_name(0), "| precision:", "bf16" if USE_BF16 else "fp16")

## 2. The training data

The training set is a spreadsheet of 120 simulated grade 8 science and mathematics items. Each row has a subject, a standard code, a topic, a stem, four options, the key letter and a one-sentence rationale.

The cell below turns each row into a training record. The prompt is built from the subject, standard and topic. The completion is the stem, options, answer and rationale written out in the house format.

The completions have to be identical in shape: a `Stem:` line, four option lines, an `Answer:` line, a `Rationale:` line, every time. 

For teaching format and style, somewhere between 100 and 500 records is enough. With far fewer, the model tends to reproduce the specific stems it saw instead of the pattern.

In [ ]:
# STEP 3 of 9: read the items from the spreadsheet and turn each row into a training pair.
import random
import pandas as pd

items = pd.read_excel("workshop_items.xlsx")      # columns: Subject, Standard, Topic, Stem, A, B, C, D, Answer, Rationale
LETTERS = "ABCD"

# The instruction the model will be given
PROMPT = (
    "Write one grade 8 {Subject} multiple-choice item.\n"
    "Standard: {Standard}\n"
    "Topic: {Topic}\n"
    "Use the house format exactly: a line beginning 'Stem:', four option lines 'A.' through 'D.', "
    "a line beginning 'Answer:', and a line beginning 'Rationale:'.\n"
)

# and the exact layout it should produce. This is what is learnt.
HOUSE_FORMAT = (
    "Stem: {Stem}\n"
    "A. {A}\n"
    "B. {B}\n"
    "C. {C}\n"
    "D. {D}\n"
    "Answer: {Answer}\n"
    "Rationale: {Rationale}"
)

records = [{"prompt": PROMPT.format(**row), "completion": HOUSE_FORMAT.format(**row)}
           for row in items.to_dict("records")]
random.Random(8).shuffle(records)

print(len(records), "records | keys:", items["Answer"].value_counts().sort_index().to_dict())
print("\n----- sample record -----")
print(records[0]["prompt"] + records[0]["completion"])

In [ ]:
# STEP 4 of 9: hold out 15% for validation.
from datasets import Dataset

split = Dataset.from_list(records).train_test_split(test_size=0.15, seed=8)
train_ds = split["train"]
eval_ds = split["test"]
print("train:", len(train_ds), "records | eval:", len(eval_ds), "records")

## 3. The base model

Teaching a model a format and a house style does not need a large model. It needs one that already writes fluent English and is small enough to train on the processing power available, which this case is a free Colab T4. `Qwen/Qwen3-0.6B` is the model we would be using for this workshop.

A larger model writes better stems and rationales, and costs more to train and to serve. `Qwen3-1.7B` would run about three times slower on the same GPU; models of 7B and above need QLoRA to fit at all. To choose a base for your own work: if you will run the model yourself, filter the Hugging Face Hub for text-generation models with a licence you can use and a size your hardware can train; if you will use a hosted service (OpenAI, Google, Bedrock, Azure), the candidates are whatever that provider offers for fine-tuning.

## 4. Training

One behavior to know about: for a prompt-completion dataset, TRL computes the loss on the completion only. The model learns to produce your format given an instruction, without also learning to reproduce the instruction itself.

In [ ]:
# STEP 5 of 9: choose the base model and set the training options.
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

BASE_MODEL = "Qwen/Qwen3-0.6B"          
ADAPTER_DIR = "itemstyle-lora-adapter"  # where the trained adapter is saved

# LoRA settings. See the table in section 1.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

# Training settings. The first three are the ones you would change for your own data.
sft_config = SFTConfig(
    output_dir="qwen3-itemstyle-run",
    num_train_epochs=3,                  # passes over the data; more than 3 starts memorizing
    learning_rate=2e-4,                  # LoRA tolerates a higher rate than full fine-tuning
    per_device_train_batch_size=2,       # small, because T4 memory is the limit
    gradient_accumulation_steps=4,       # effective batch size = 2 x 4 = 8
    max_length=512,                      # tokens per record; the example are well under this
    bf16=USE_BF16,                       # bf16 on newer GPUs, fp16 on a T4
    fp16=not USE_BF16,
    eval_strategy="steps",               # evaluate every 10 steps so we can watch for overfitting
    eval_steps=10,
    logging_steps=5,
    save_strategy="no",                  # we save the adapter ourselves in the next cell
    report_to="none",
)

# SFTTrainer downloads the model, attaches the LoRA adapter, tokenizes the records,
# and masks the prompt out of the loss so only the completion is learned.
trainer = SFTTrainer(
    model=BASE_MODEL,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=sft_config,
    peft_config=lora_config,
)
print("Trainer ready.")

In [ ]:
# STEP 6 of 9: train, then save the adapter. About 3 to 6 minutes on a T4.
# Training loss should fall steadily. Eval loss should fall, then flatten; if it climbs while
# training loss keeps falling, the model is memorizing and you want fewer epochs.
train_result = trainer.train()
trainer.save_model(ADAPTER_DIR)
eval_loss = trainer.evaluate()["eval_loss"]
print(f"\nfinal training loss {train_result.training_loss:.3f} | eval loss {eval_loss:.3f} | adapter saved to {ADAPTER_DIR}/")

### What was saved

The adapter is a few tens of megabytes. The base model it modifies is about 1.2 GB, and the adapter does nothing without that exact base.

## 5. Before and after

Lets do a comparison of adapter on vs adapter off with the same base weights, same prompt, and same decoding settings. PEFT provides `disable_adapter()`, a context manager that temporarily switches the adapter out, so this is the same model in both conditions.

In [ ]:
# STEP 7 of 9: generate items from the base model and from the tuned model
tokenizer = trainer.processing_class
model = trainer.model
model.eval()


def generate(prompt, max_new_tokens=180):
    """Ask the model to continue the prompt. Greedy decoding, so runs are repeatable."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def parse_item(text):
    """Pull each house-format field out of a generation. A missing field is left empty."""
    def field(label):
        for line in text.split("\n"):
            if line.startswith(label):
                return line[len(label):].strip()
        return ""

    fields = {
        "Stem": field("Stem:"),
        "A": field("A."), "B": field("B."), "C": field("C."), "D": field("D."),
        "Answer": field("Answer:")[:1],
        "Rationale": field("Rationale:"),
    }
    required = [fields[k] for k in ("Stem", "A", "B", "C", "D", "Rationale")]
    fields["Format OK"] = all(required) and fields["Answer"] in LETTERS
    return fields


# Standards the model was NOT trained on.
TEST_STANDARDS = [
    ("science", "MS-ESS1-4 (rock strata and Earth's history)", "relative dating of rock layers"),
    ("science", "MS-LS3-2 (sexual and asexual reproduction)", "genetic variation in offspring"),
    ("mathematics", "8.G.A.1 (rigid transformations)", "properties of reflections"),
    ("mathematics", "8.NS.A.2 (irrational numbers)", "estimating square roots"),
]

rows = []
for subject, standard, topic in TEST_STANDARDS:
    prompt = PROMPT.format(Subject=subject, Standard=standard, Topic=topic)
    with model.disable_adapter():          # same weights, adapter switched off = the untouched base model
        base_output = generate(prompt)
    tuned_output = generate(prompt)        # adapter on
    rows.append({"Standard": standard, "Condition": "base", **parse_item(base_output), "Raw output": base_output})
    rows.append({"Standard": standard, "Condition": "adapter", **parse_item(tuned_output), "Raw output": tuned_output})

results = pd.DataFrame(rows)
print(results.groupby("Condition")["Format OK"].sum().rename("items in house format"))
print("\n----- base model, first standard -----\n" + rows[0]["Raw output"])
print("\n----- tuned model, first standard -----\n" + rows[1]["Raw output"])

### What to look for

Look at the form of the output. Did the adapted output land in the house format? Are there exactly four option lines, labeled the right way? Is there an `Answer:` line and a `Rationale:` line, in that order, with nothing extra? Does the register match: the same sentence length, the same directness, the same absence of "Which of the following best describes"?

If the "after" output looks worse rather than different (repeating itself, truncating, drifting into a second item), the usual cause is too many epochs on too little data. Drop to 2 epochs, or add more items.

In [ ]:
# STEP 8 of 9: write the results to one Excel workbook.
#   Sheet "Items": one row per generated item, base and adapter side by side
#   Sheet "Run":   the model, settings and losses, so the workbook records how the items were made
run = pd.DataFrame([
    ["Base model", BASE_MODEL],
    ["Adapter", ADAPTER_DIR],
    ["Training records", len(train_ds)],
    ["Eval records", len(eval_ds)],
    ["LoRA rank", lora_config.r],
    ["LoRA alpha", lora_config.lora_alpha],
    ["Epochs", sft_config.num_train_epochs],
    ["Learning rate", sft_config.learning_rate],
    ["Final training loss", round(train_result.training_loss, 4)],
    ["Final eval loss", round(eval_loss, 4)],
], columns=["Setting", "Value"])

OUTPUT_FILE = "itemgen_finetune_results.xlsx"
with pd.ExcelWriter(OUTPUT_FILE) as writer:
    results.to_excel(writer, sheet_name="Items", index=False)
    run.to_excel(writer, sheet_name="Run", index=False)
print("Saved", OUTPUT_FILE)

try:                                   # in Colab, also download it
    from google.colab import files
    files.download(OUTPUT_FILE)
except ImportError:
    pass

## 6. Generate items with the tuned model

The comparison above used four prompts. This step is the one you would run to actually produce items: set how many you want, list the standards to write for, and the tuned model writes them into a spreadsheet, one row per item, with the parsed fields and a Format OK flag. Sampling is switched on here so the same standard produces a different item each time; the comparison in step 7 used greedy decoding so that its before-and-after was repeatable.

These are drafts. Every row still goes through content review, and the Format OK column only says the item is in the right shape.

In [ ]:
# STEP 9 of 9: generate a batch of items with the tuned model and save them to Excel.
N_ITEMS = 20                              # how many items to generate
OUTPUT_FILE = "generated_items.xlsx"

# Standards to write for. The loop cycles through this list until N_ITEMS is reached.
STANDARDS = [
    ("science", "MS-ESS1-4 (rock strata and Earth's history)", "relative dating of rock layers"),
    ("science", "MS-LS3-2 (sexual and asexual reproduction)", "genetic variation in offspring"),
    ("mathematics", "8.G.A.1 (rigid transformations)", "properties of reflections"),
    ("mathematics", "8.NS.A.2 (irrational numbers)", "estimating square roots"),
]

generated = []
for i in range(N_ITEMS):
    subject, standard, topic = STANDARDS[i % len(STANDARDS)]
    prompt = PROMPT.format(Subject=subject, Standard=standard, Topic=topic)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():                 # sampling, so repeated standards give different items
        output = model.generate(**inputs, max_new_tokens=180, do_sample=True, temperature=0.7, top_p=0.9)
    text = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    generated.append({"Item": i + 1, "Subject": subject, "Standard": standard, "Topic": topic,
                      **parse_item(text), "Raw output": text})

generated = pd.DataFrame(generated)
generated.to_excel(OUTPUT_FILE, index=False)
print(f"{N_ITEMS} items generated, {int(generated['Format OK'].sum())} in house format. Saved {OUTPUT_FILE}")

try:                                      # in Colab, also download it
    from google.colab import files
    files.download(OUTPUT_FILE)
except ImportError:
    pass

#### Further reading

**Papers**
- LoRA (Hu et al., 2021): https://arxiv.org/abs/2106.09685
- QLoRA (Dettmers et al., 2023): https://arxiv.org/abs/2305.14314
- InstructGPT, instruction tuning and RLHF (Ouyang et al., 2022): https://arxiv.org/abs/2203.02155
- Prompt tuning (Lester et al., 2021): https://arxiv.org/abs/2104.08691
- Adapters (Houlsby et al., 2019): https://arxiv.org/abs/1902.00751

**Documentation for the libraries this notebook uses**
- TRL SFTTrainer, including the prompt-completion format: https://huggingface.co/docs/trl/sft_trainer
- TRL quickstart: https://huggingface.co/docs/trl/quickstart
- TRL with PEFT adapters: https://huggingface.co/docs/trl/peft_integration
- PEFT LoRA guide: https://huggingface.co/docs/peft/developer_guides/lora
- Transformers: https://huggingface.co/docs/transformers
- Finding base models: https://huggingface.co/models?pipeline_tag=text-generation

**Hosted fine-tuning (check the data terms before uploading items)**
- OpenAI, supervised fine-tuning guide: https://developers.openai.com/api/docs/guides/supervised-fine-tuning
- Google Cloud, tuning Gemini and open models: https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/tuning
- Amazon Bedrock, custom models: https://docs.aws.amazon.com/bedrock/latest/userguide/custom-models.html
- Microsoft Foundry (Azure), fine-tuning overview: https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/fine-tuning-overview